<a href="https://colab.research.google.com/github/Mridul33/capstone-project/blob/main/Stage_3.1-3.3_RE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Stage 3.3: PubMedBERT Relation Extraction Fine-Tuning

This notebook continues from Stage 3.2, where the BioRED relation extraction examples were prepared, entity markers were added, relation labels were encoded, and the processed train, development, and test datasets were saved. In this stage, those prepared outputs are loaded and used to fine-tune PubMedBERT for multiclass relation classification. The workflow includes final dataset validation, handling sequences longer than 512 tokens, model training, class-weighted loss to address label imbalance, evaluation, error analysis, and saving the final predictions for knowledge graph construction.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import pickle

save_directory = "/content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs"

print("Loading files from:", save_directory)
print("Folder exists:", os.path.exists(save_directory))

Loading files from: /content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs
Folder exists: True


In [3]:
print("Files in the saved-output folder:")

for file_name in os.listdir(save_directory):
    print("-", file_name)

Files in the saved-output folder:
- biored_prepared_relation_data.pkl
- biored_relation_label_mappings.pkl


In [4]:
re_data_path = os.path.join(
    save_directory,
    "biored_prepared_relation_data.pkl"
)

mapping_path = os.path.join(
    save_directory,
    "biored_relation_label_mappings.pkl"
)

with open(re_data_path, "rb") as file:
    loaded_re_data = pickle.load(file)

with open(mapping_path, "rb") as file:
    loaded_relation_mappings = pickle.load(file)

print("Saved files loaded successfully.")

Saved files loaded successfully.


In [5]:
prepared_train_re = loaded_re_data["train"]
prepared_dev_re = loaded_re_data["development"]
prepared_test_re = loaded_re_data["test"]

relation_label_to_id = loaded_relation_mappings["label_to_id"]
relation_id_to_label = loaded_relation_mappings["id_to_label"]

In [6]:
print("Training examples:", len(prepared_train_re))
print("Development examples:", len(prepared_dev_re))
print("Test examples:", len(prepared_test_re))

print("\nNumber of relation classes:", len(relation_label_to_id))

print("\nRelation label mapping:")
print(relation_label_to_id)

Training examples: 8356
Development examples: 2324
Test examples: 2326

Number of relation classes: 9

Relation label mapping:
{'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}


In [7]:
print("First training example:")
print(prepared_train_re[0])

print("\nAvailable fields:")
print(prepared_train_re[0].keys())

First training example:
{'document_id': '26115410', 'text': 'Mechanisms Underlying Latent Disease Risk Associated with Early-Life Arsenic Exposure: Current Research Trends and Scientific Gaps. BACKGROUND: Millions of individuals worldwide, particularly those living in rural and developing areas, are exposed to harmful levels of inorganic arsenic (iAs) in their drinking water. Inorganic As exposure during key developmental periods is associated with a variety of adverse health effects including those that are evident in adulthood. There is considerable interest in identifying the molecular mechanisms that relate early-life iAs exposure to the development of these latent diseases, particularly in relationship to cancer. OBJECTIVES: This work summarizes research on the molecular mechanisms that underlie the increased risk of cancer development in adulthood that is associated with early-life iAs exposure. DISCUSSION: Epigenetic reprogramming that imparts functional changes in gene expressi

##Stage 3.3: PubMedBERT Relation Extraction Fine-Tuning

In [8]:
print("Training examples:", len(prepared_train_re))
print("Development examples:", len(prepared_dev_re))
print("Test examples:", len(prepared_test_re))

print("\nRelation label-to-ID mapping:")
print(relation_label_to_id)

print("\nRelation ID-to-label mapping:")
print(relation_id_to_label)

print("\nNumber of relation classes:", len(relation_label_to_id))

Training examples: 8356
Development examples: 2324
Test examples: 2326

Relation label-to-ID mapping:
{'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}

Relation ID-to-label mapping:
{0: 'Association', 1: 'Bind', 2: 'Comparison', 3: 'Conversion', 4: 'Cotreatment', 5: 'Drug_Interaction', 6: 'Negative_Correlation', 7: 'Positive_Correlation', 8: 'No_Relation'}

Number of relation classes: 9


In [9]:
sample_example = prepared_train_re[0]

print("Available fields:")
print(sample_example.keys())

print("\nDocument ID:")
print(sample_example["document_id"])

print("\nMarked text:")
print(sample_example["marked_text"])

print("\nRelation label:")
print(sample_example["relation_label"])

print("\nRelation label ID:")
print(sample_example["relation_label_id"])

print("\nInput IDs length:")
print(len(sample_example["input_ids"]))

print("\nAttention mask length:")
print(len(sample_example["attention_mask"]))

Available fields:
dict_keys(['document_id', 'text', 'entity1_identifier', 'entity1_annotation_id', 'entity1_text', 'entity1_type', 'entity1_start', 'entity1_end', 'entity2_identifier', 'entity2_annotation_id', 'entity2_text', 'entity2_type', 'entity2_start', 'entity2_end', 'relation_label', 'relation_label_id', 'marked_text', 'input_ids', 'attention_mask'])

Document ID:
26115410

Marked text:
Mechanisms Underlying Latent Disease Risk Associated with Early-Life Arsenic Exposure: Current Research Trends and Scientific Gaps. BACKGROUND: Millions of individuals worldwide, particularly those living in rural and developing areas, are exposed to harmful levels of @ChemicalEntity$ inorganic arsenic @/ChemicalEntity$ (iAs) in their drinking water. Inorganic As exposure during key developmental periods is associated with a variety of adverse health effects including those that are evident in adulthood. There is considerable interest in identifying the molecular mechanisms that relate early-life

In [10]:
from collections import Counter


def validate_re_dataset(dataset, split_name, relation_label_to_id):
    required_fields = {
        "document_id",
        "marked_text",
        "relation_label",
        "relation_label_id",
        "input_ids",
        "attention_mask"
    }

    errors = Counter()
    label_counts = Counter()

    for example in dataset:

        # Check required fields
        if not required_fields.issubset(example):
            errors["missing_fields"] += 1
            continue

        label = example["relation_label"]
        label_id = example["relation_label_id"]
        marked_text = example["marked_text"]
        input_ids = example["input_ids"]
        attention_mask = example["attention_mask"]

        label_counts[label] += 1

        # Check label and label ID
        if label not in relation_label_to_id:
            errors["invalid_labels"] += 1
        elif relation_label_to_id[label] != label_id:
            errors["label_id_mismatches"] += 1

        # Check text and tokenised inputs
        if not marked_text.strip():
            errors["empty_texts"] += 1

        if not input_ids:
            errors["empty_input_ids"] += 1

        if len(input_ids) != len(attention_mask):
            errors["length_mismatches"] += 1

        if any(value not in (0, 1) for value in attention_mask):
            errors["invalid_attention_masks"] += 1

        # Check both entity-marker pairs
        if not all(marker in marked_text for marker in ["@", "@/", "#", "#/"]):
            errors["missing_entity_markers"] += 1

    positive_count = sum(
        count
        for label, count in label_counts.items()
        if label != "No_Relation"
    )

    negative_count = label_counts.get("No_Relation", 0)

    sequence_lengths = [
        len(example["input_ids"])
        for example in dataset
        if "input_ids" in example
    ]

    print(f"\n{split_name.upper()} VALIDATION")
    print("-" * 40)
    print("Total examples:", len(dataset))
    print("Positive examples:", positive_count)
    print("No_Relation examples:", negative_count)
    print("Minimum sequence length:", min(sequence_lengths))
    print("Maximum sequence length:", max(sequence_lengths))
    print(
        "Sequences longer than 512:",
        sum(length > 512 for length in sequence_lengths)
    )

    print("\nErrors:")
    for error_name in [
        "missing_fields",
        "invalid_labels",
        "label_id_mismatches",
        "empty_texts",
        "empty_input_ids",
        "length_mismatches",
        "invalid_attention_masks",
        "missing_entity_markers"
    ]:
        print(f"{error_name}: {errors[error_name]}")

    total_errors = sum(errors.values())

    if total_errors == 0:
        print("\n✅ Dataset passed validation.")
    else:
        print(f"\n❌ Total validation errors: {total_errors}")

    return {
        "errors": errors,
        "label_counts": label_counts,
        "positive_count": positive_count,
        "negative_count": negative_count,
        "total_errors": total_errors
    }

In [11]:
train_validation = validate_re_dataset(
    prepared_train_re,
    "Training",
    relation_label_to_id
)

dev_validation = validate_re_dataset(
    prepared_dev_re,
    "Development",
    relation_label_to_id
)

test_validation = validate_re_dataset(
    prepared_test_re,
    "Test",
    relation_label_to_id
)


TRAINING VALIDATION
----------------------------------------
Total examples: 8356
Positive examples: 4178
No_Relation examples: 4178
Minimum sequence length: 76
Maximum sequence length: 760
Sequences longer than 512: 334

Errors:
missing_fields: 0
invalid_labels: 0
label_id_mismatches: 0
empty_texts: 0
empty_input_ids: 0
length_mismatches: 0
invalid_attention_masks: 0
missing_entity_markers: 0

✅ Dataset passed validation.

DEVELOPMENT VALIDATION
----------------------------------------
Total examples: 2324
Positive examples: 1162
No_Relation examples: 1162
Minimum sequence length: 176
Maximum sequence length: 647
Sequences longer than 512: 190

Errors:
missing_fields: 0
invalid_labels: 0
label_id_mismatches: 0
empty_texts: 0
empty_input_ids: 0
length_mismatches: 0
invalid_attention_masks: 0
missing_entity_markers: 0

✅ Dataset passed validation.

TEST VALIDATION
----------------------------------------
Total examples: 2326
Positive examples: 1163
No_Relation examples: 1163
Minimum se

##Stage 3.3B — Handle sequences longer than 512 tokens
For relation extraction, ordinary right-side truncation is risky because it may remove one of the marked entities. Instead, I used entity-preserving cropping so that both marked entities remained in the input while retaining as much surrounding context as possible.

In [12]:
long_train_examples = [
    example
    for example in prepared_train_re
    if len(example["input_ids"]) > 512
]

print("Number of long training examples:", len(long_train_examples))

sample_long_example = long_train_examples[0]

print("\nDocument ID:", sample_long_example["document_id"])
print("Relation label:", sample_long_example["relation_label"])
print("Sequence length:", len(sample_long_example["input_ids"]))

print("\nMarked text:")
print(sample_long_example["marked_text"])

Number of long training examples: 334

Document ID: 24623966
Relation label: No_Relation
Sequence length: 710

Marked text:
Endothelial NADPH oxidase 4 mediates @GeneOrGeneProduct$ vascular endothelial growth factor receptor 2 @/GeneOrGeneProduct$-induced intravitreal neovascularization in a rat model of retinopathy of prematurity. PURPOSE: #GeneOrGeneProduct$ NADPH oxidase #/GeneOrGeneProduct$-generated reactive oxygen species (ROS) are implicated in angiogenesis. Isoforms of NADPH oxidase NOX1, NOX2, and NOX4 are reported to be expressed in endothelial cells (ECs). Of these, NOX1 and NOX2 have been reported to contribute to intravitreal neovascularization (IVNV) in oxygen-induced retinopathy (OIR) models. In this study, we tested the hypothesis that the isoform NOX4 in ECs contributed to vascular endothelial growth factor (VEGF)-induced angiogenesis and IVNV. METHODS: Isoforms of NADPH oxidase MRNA were measured in several types of cultured vascular ecs: human retinal microvascular E

In [13]:
!pip install -q transformers

In [14]:
from transformers import AutoTokenizer

model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

print("Tokenizer loaded successfully.")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Maximum model length:", tokenizer.model_max_length)
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

Tokenizer loaded successfully.
Tokenizer class: BertTokenizer
Maximum model length: 1000000000000000019884624838656
Vocabulary size: 28895


In [15]:
MAX_LENGTH = 512

print("Maximum sequence length:", MAX_LENGTH)

Maximum sequence length: 512


In [16]:
def find_complete_marker_pair(tokens, marker_symbol):
    markers = []

    for start_index, token in enumerate(tokens):
        if token != marker_symbol:
            continue

        search_limit = min(start_index + 20, len(tokens))

        for end_index in range(start_index + 1, search_limit):
            if tokens[end_index] == "$":
                marker_tokens = tokens[start_index:end_index + 1]

                markers.append({
                    "start": start_index,
                    "end": end_index,
                    "is_closing": "/" in marker_tokens
                })
                break

    opening_markers = [
        marker for marker in markers
        if not marker["is_closing"]
    ]

    closing_markers = [
        marker for marker in markers
        if marker["is_closing"]
    ]

    if len(opening_markers) != 1 or len(closing_markers) != 1:
        raise ValueError(
            f"Expected one opening and one closing "
            f"'{marker_symbol}' marker, but found "
            f"{len(opening_markers)} opening and "
            f"{len(closing_markers)} closing."
        )

    return opening_markers[0], closing_markers[0]

In [17]:
def find_invalid_marker_examples(dataset, split_name, tokenizer):
    invalid_examples = []

    for index, example in enumerate(dataset):
        tokens = tokenizer.convert_ids_to_tokens(
            example["input_ids"]
        )

        problems = []

        try:
            find_complete_marker_pair(tokens, "@")
        except ValueError:
            problems.append("entity1_marker_problem")

        try:
            find_complete_marker_pair(tokens, "#")
        except ValueError:
            problems.append("entity2_marker_problem")

        if problems:
            invalid_examples.append({
                "index": index,
                "document_id": example["document_id"],
                "relation_label": example["relation_label"],
                "sequence_length": len(example["input_ids"]),
                "problems": problems
            })

    print(f"\n{split_name.upper()}")
    print("Total examples:", len(dataset))
    print("Invalid marker examples:", len(invalid_examples))

    if invalid_examples:
        print("\nFirst five:")
        for issue in invalid_examples[:5]:
            print(issue)

    return invalid_examples

In [18]:
invalid_train_examples = find_invalid_marker_examples(
    prepared_train_re,
    "Training",
    tokenizer
)

invalid_dev_examples = find_invalid_marker_examples(
    prepared_dev_re,
    "Development",
    tokenizer
)

invalid_test_examples = find_invalid_marker_examples(
    prepared_test_re,
    "Test",
    tokenizer
)


TRAINING
Total examples: 8356
Invalid marker examples: 16

First five:
{'index': 128, 'document_id': '22303482', 'relation_label': 'Negative_Correlation', 'sequence_length': 512, 'problems': ['entity2_marker_problem']}
{'index': 172, 'document_id': '28472177', 'relation_label': 'Association', 'sequence_length': 512, 'problems': ['entity2_marker_problem']}
{'index': 2420, 'document_id': '28472177', 'relation_label': 'Association', 'sequence_length': 512, 'problems': ['entity2_marker_problem']}
{'index': 2606, 'document_id': '17379047', 'relation_label': 'Negative_Correlation', 'sequence_length': 512, 'problems': ['entity1_marker_problem']}
{'index': 2974, 'document_id': '15200408', 'relation_label': 'Association', 'sequence_length': 512, 'problems': ['entity1_marker_problem']}

DEVELOPMENT
Total examples: 2324
Invalid marker examples: 11

First five:
{'index': 89, 'document_id': '24036311', 'relation_label': 'Bind', 'sequence_length': 302, 'problems': ['entity1_marker_problem']}
{'inde

In [19]:
invalid_train_indices = {
    item["index"] for item in invalid_train_examples
}

invalid_dev_indices = {
    item["index"] for item in invalid_dev_examples
}

invalid_test_indices = {
    item["index"] for item in invalid_test_examples
}

In [20]:
clean_train_re = [
    example
    for index, example in enumerate(prepared_train_re)
    if index not in invalid_train_indices
]

clean_dev_re = [
    example
    for index, example in enumerate(prepared_dev_re)
    if index not in invalid_dev_indices
]

clean_test_re = [
    example
    for index, example in enumerate(prepared_test_re)
    if index not in invalid_test_indices
]

print("Clean training size:", len(clean_train_re))
print("Clean development size:", len(clean_dev_re))
print("Clean test size:", len(clean_test_re))

Clean training size: 8340
Clean development size: 2313
Clean test size: 2300


In [21]:
def show_length_summary(dataset, split_name):
    lengths = [len(example["input_ids"]) for example in dataset]

    print(f"\n{split_name.upper()}")
    print("Total examples:", len(dataset))
    print("Minimum length:", min(lengths))
    print("Maximum length:", max(lengths))
    print(
        "Examples longer than 512:",
        sum(length > 512 for length in lengths)
    )

In [22]:
show_length_summary(clean_train_re, "Clean Training")
show_length_summary(clean_dev_re, "Clean Development")
show_length_summary(clean_test_re, "Clean Test")


CLEAN TRAINING
Total examples: 8340
Minimum length: 76
Maximum length: 760
Examples longer than 512: 334

CLEAN DEVELOPMENT
Total examples: 2313
Minimum length: 176
Maximum length: 647
Examples longer than 512: 190

CLEAN TEST
Total examples: 2300
Minimum length: 118
Maximum length: 586
Examples longer than 512: 283


In [23]:
def crop_valid_relation_example(example, tokenizer, max_length=512):
    input_ids = example["input_ids"]

    if len(input_ids) <= max_length:
        result = example.copy()
        result["was_cropped"] = False
        result["cropping_method"] = "none"
        return result

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    entity1_open, entity1_close = find_complete_marker_pair(tokens, "@")
    entity2_open, entity2_close = find_complete_marker_pair(tokens, "#")

    entity_spans = sorted(
        [
            (entity1_open["start"], entity1_close["end"]),
            (entity2_open["start"], entity2_close["end"])
        ],
        key=lambda span: span[0]
    )

    first_start, first_end = entity_spans[0]
    second_start, second_end = entity_spans[1]

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    continuous_budget = max_length - 2
    full_region_length = second_end - first_start + 1

    # Case 1: both entities fit inside one continuous window
    if full_region_length <= continuous_budget:
        remaining_context = continuous_budget - full_region_length

        context_before = remaining_context // 2
        context_after = remaining_context - context_before

        window_start = max(1, first_start - context_before)
        window_end = min(
            len(input_ids) - 1,
            second_end + context_after + 1
        )

        while window_end - window_start < continuous_budget:
            if window_start > 1:
                window_start -= 1
            elif window_end < len(input_ids) - 1:
                window_end += 1
            else:
                break

        cropped_ids = (
            [cls_id]
            + input_ids[window_start:window_end]
            + [sep_id]
        )

        method = "single_window"

    # Case 2: entities are too far apart
    else:
        dual_budget = max_length - 3

        first_entity_length = first_end - first_start + 1
        second_entity_length = second_end - second_start + 1

        required_tokens = (
            first_entity_length + second_entity_length
        )

        if required_tokens > dual_budget:
            raise ValueError(
                f"Entity spans exceed token budget in "
                f"document {example['document_id']}."
            )

        available_context = dual_budget - required_tokens

        first_context = available_context // 2
        second_context = available_context - first_context

        def build_window(entity_start, entity_end, context_budget):
            entity_length = entity_end - entity_start + 1
            target_length = entity_length + context_budget

            before = context_budget // 2
            after = context_budget - before

            start = max(1, entity_start - before)
            end = min(
                len(input_ids) - 1,
                entity_end + after + 1
            )

            while end - start < target_length:
                if start > 1:
                    start -= 1
                elif end < len(input_ids) - 1:
                    end += 1
                else:
                    break

            return input_ids[start:end]

        first_window = build_window(
            first_start,
            first_end,
            first_context
        )

        second_window = build_window(
            second_start,
            second_end,
            second_context
        )

        cropped_ids = (
            [cls_id]
            + first_window
            + [sep_id]
            + second_window
            + [sep_id]
        )

        method = "dual_window"

    if len(cropped_ids) > max_length:
        raise ValueError(
            f"Cropped sequence exceeds {max_length} tokens "
            f"in document {example['document_id']}."
        )

    result = example.copy()
    result["input_ids"] = cropped_ids
    result["attention_mask"] = [1] * len(cropped_ids)
    result["was_cropped"] = True
    result["cropping_method"] = method
    result["original_sequence_length"] = len(input_ids)
    result["cropped_sequence_length"] = len(cropped_ids)

    return result

In [24]:
final_train_re = [
    crop_valid_relation_example(example, tokenizer, MAX_LENGTH)
    for example in clean_train_re
]

final_dev_re = [
    crop_valid_relation_example(example, tokenizer, MAX_LENGTH)
    for example in clean_dev_re
]

final_test_re = [
    crop_valid_relation_example(example, tokenizer, MAX_LENGTH)
    for example in clean_test_re
]

print("Entity-preserving cropping completed.")

Entity-preserving cropping completed.


In [25]:
from collections import Counter

def validate_final_re_dataset(dataset, split_name, tokenizer):
    method_counts = Counter()
    overlength_count = 0
    mask_error_count = 0
    marker_error_count = 0

    for example in dataset:
        method_counts[example["cropping_method"]] += 1

        if len(example["input_ids"]) > MAX_LENGTH:
            overlength_count += 1

        if len(example["input_ids"]) != len(example["attention_mask"]):
            mask_error_count += 1

        tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

        try:
            find_complete_marker_pair(tokens, "@")
            find_complete_marker_pair(tokens, "#")
        except ValueError:
            marker_error_count += 1

    print(f"\n{split_name.upper()}")
    print("Total examples:", len(dataset))
    print("Unchanged:", method_counts["none"])
    print("Single-window cropped:", method_counts["single_window"])
    print("Dual-window cropped:", method_counts["dual_window"])

    print("\nExamples longer than 512:", overlength_count)
    print("Input/mask mismatches:", mask_error_count)
    print("Marker errors:", marker_error_count)

    passed = (
        overlength_count == 0
        and mask_error_count == 0
        and marker_error_count == 0
    )

    if passed:
        print("\nDataset passed final validation.")
    else:
        print("\nDataset requires inspection.")

    return passed

In [26]:
train_final_validation = validate_final_re_dataset(
    final_train_re,
    "Final Training",
    tokenizer
)

dev_final_validation = validate_final_re_dataset(
    final_dev_re,
    "Final Development",
    tokenizer
)

test_final_validation = validate_final_re_dataset(
    final_test_re,
    "Final Test",
    tokenizer
)


FINAL TRAINING
Total examples: 8340
Unchanged: 8006
Single-window cropped: 330
Dual-window cropped: 4

Examples longer than 512: 0
Input/mask mismatches: 0
Marker errors: 0

Dataset passed final validation.

FINAL DEVELOPMENT
Total examples: 2313
Unchanged: 2123
Single-window cropped: 188
Dual-window cropped: 2

Examples longer than 512: 0
Input/mask mismatches: 0
Marker errors: 0

Dataset passed final validation.

FINAL TEST
Total examples: 2300
Unchanged: 2017
Single-window cropped: 281
Dual-window cropped: 2

Examples longer than 512: 0
Input/mask mismatches: 0
Marker errors: 0

Dataset passed final validation.


### Stage 3.3C: Preparing Hugging Face Datasets and Dynamic Padding

After cleaning and validating the relation extraction data, the final training,
development, and test examples are converted into Hugging Face Dataset objects.

Only the model-required fields are retained:

- `input_ids`
- `attention_mask`
- `labels`

The existing `relation_label_id` value is renamed to `labels` because this is the
field expected by the Hugging Face sequence-classification model and Trainer.

Because the input sequences have different lengths, dynamic padding is used.
This pads each batch only to the length of the longest sequence in that batch,
rather than padding every example to the maximum length of 512 tokens. This
reduces unnecessary padding and makes training more memory-efficient.

In [27]:
!pip install -q datasets transformers evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [28]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

In [29]:
def convert_to_hf_dataset(dataset):
    return Dataset.from_dict({
        "input_ids": [
            example["input_ids"]
            for example in dataset
        ],
        "attention_mask": [
            example["attention_mask"]
            for example in dataset
        ],
        "labels": [
            example["relation_label_id"]
            for example in dataset
        ]
    })

In [30]:
train_dataset = convert_to_hf_dataset(final_train_re)
dev_dataset = convert_to_hf_dataset(final_dev_re)
test_dataset = convert_to_hf_dataset(final_test_re)

print("Training dataset:", train_dataset)
print("Development dataset:", dev_dataset)
print("Test dataset:", test_dataset)

Training dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8340
})
Development dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2313
})
Test dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2300
})


In [31]:
print(train_dataset[0])

{'input_ids': [2, 3138, 4443, 10079, 2174, 2209, 2138, 1715, 2743, 16, 2978, 11513, 3035, 29, 3040, 2698, 7472, 1690, 6993, 11491, 17, 2645, 29, 21665, 1685, 3415, 7525, 15, 4124, 2415, 5126, 1682, 7016, 1690, 4510, 3655, 15, 1810, 4183, 1701, 12910, 2182, 1685, 35, 3829, 1865, 2677, 7, 10326, 11513, 35, 18, 3829, 1865, 2677, 7, 11, 27525, 12, 1682, 2076, 7055, 2972, 17, 10326, 1732, 3035, 2098, 3784, 5638, 5904, 1744, 2138, 1715, 42, 4692, 1685, 4246, 2161, 2228, 2574, 2415, 1760, 1810, 7460, 1682, 10759, 17, 2100, 1744, 6394, 4826, 1682, 6656, 1680, 2894, 3138, 1760, 11832, 2743, 16, 2978, 27525, 3035, 1701, 1680, 2418, 1685, 1898, 10079, 3316, 15, 4124, 1682, 2953, 1701, 6, 2174, 2588, 23162, 2677, 17511, 7335, 2327, 7, 2310, 6, 18, 2174, 2588, 23162, 2677, 17511, 7335, 2327, 7, 17, 4283, 29, 1805, 2626, 11626, 2698, 1755, 1680, 2894, 3138, 1760, 13481, 1680, 2165, 2209, 1685, 2310, 2418, 1682, 10759, 1760, 1744, 2138, 1715, 2743, 16, 2978, 27525, 3035, 17, 6372, 29, 9693, 19882, 17

In [32]:
print(
    "Input IDs length:",
    len(train_dataset[0]["input_ids"])
)

print(
    "Attention mask length:",
    len(train_dataset[0]["attention_mask"])
)

print(
    "Label:",
    train_dataset[0]["labels"]
)

Input IDs length: 256
Attention mask length: 256
Label: 7


In [33]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

print("Dynamic padding collator created.")

Dynamic padding collator created.


In [34]:
sample_batch = [
    train_dataset[index]
    for index in range(4)
]

padded_batch = data_collator(sample_batch)

print("Batch keys:", padded_batch.keys())
print(
    "Input IDs shape:",
    padded_batch["input_ids"].shape
)
print(
    "Attention mask shape:",
    padded_batch["attention_mask"].shape
)
print(
    "Labels shape:",
    padded_batch["labels"].shape
)

Batch keys: KeysView({'input_ids': tensor([[    2,  3138,  4443,  ...,     0,     0,     0],
        [    2, 23407,    16,  ...,     0,     0,     0],
        [    2,    35,  2397,  ...,     0,     0,     0],
        [    2, 24720,    16,  ...,  1830,    17,     3]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([7, 8, 0, 8])})
Input IDs shape: torch.Size([4, 363])
Attention mask shape: torch.Size([4, 363])
Labels shape: torch.Size([4])


In [35]:
print("Maximum padded length:", padded_batch["input_ids"].shape[1])

print(
    "Input and attention-mask shapes match:",
    padded_batch["input_ids"].shape
    == padded_batch["attention_mask"].shape
)

print(
    "Batch size matches labels:",
    padded_batch["input_ids"].shape[0]
    == padded_batch["labels"].shape[0]
)

Maximum padded length: 363
Input and attention-mask shapes match: True
Batch size matches labels: True


### Stage 3.3D: Loading PubMedBERT for Relation Classification

PubMedBERT is now loaded as a sequence-classification model for the biomedical
relation extraction task. Unlike the NER model, which predicts one label for
each token, the relation extraction model predicts one relation label for the
entire entity-marked input sequence.

The classification layer is configured with nine output classes representing
the eight BioRED relation types and the `No_Relation` class. The saved relation
label mappings are supplied to the model so that prediction IDs can be converted
back into meaningful relation names.

In [36]:
from transformers import AutoModelForSequenceClassification

num_relation_labels = len(relation_label_to_id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_relation_labels,
    label2id=relation_label_to_id,
    id2label=relation_id_to_label
)

print("Model loaded successfully.")
print("Model name:", model_name)
print("Number of relation classes:", model.config.num_labels)
print("Problem type:", model.config.problem_type)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

Model loaded successfully.
Model name: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Number of relation classes: 9
Problem type: None


In [37]:
print("Label-to-ID mapping:")
print(model.config.label2id)

print("\nID-to-label mapping:")
print(model.config.id2label)

Label-to-ID mapping:
{'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}

ID-to-label mapping:
{0: 'Association', 1: 'Bind', 2: 'Comparison', 3: 'Conversion', 4: 'Cotreatment', 5: 'Drug_Interaction', 6: 'Negative_Correlation', 7: 'Positive_Correlation', 8: 'No_Relation'}


In [38]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L4


### Stage 3.3E: Defining Evaluation Metrics

The relation extraction model will be evaluated using accuracy, precision,
recall, and F1-score.

Because this is a multiclass classification task with nine relation labels,
macro-averaged precision, recall, and F1-score are calculated. Macro averaging
computes the metric separately for each relation class and then gives equal
importance to every class, including the less frequent relation types.

Weighted F1-score is also calculated to provide an overall result that considers
the number of examples in each class. The macro F1-score is used as the main
metric for selecting the best model during training.

In [39]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1
    }

In [40]:
print("Evaluation metrics function created.")

Evaluation metrics function created.


### Stage 3.3F: Configuring the Training Process

The Hugging Face Trainer is configured to fine-tune PubMedBERT on the prepared
relation extraction dataset.

Training will use a learning rate of `2e-5`, a batch size of `4`, and `3`
training epochs, matching the main settings used during NER fine-tuning.
Evaluation and checkpoint saving will be performed after each epoch.

The best model will be selected using the development-set macro F1-score.
Macro F1 is used because it gives equal importance to all nine relation classes,
including less frequent relation types. Mixed-precision training is enabled on
the GPU to reduce memory usage and improve training speed.

In [41]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/pubmedbert_re_results",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,

    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=100,

    save_total_limit=2,

    report_to="none"
)

print("Training arguments created.")

Training arguments created.


### Stage 3.3G: Creating the Hugging Face Trainer

The Hugging Face Trainer combines the model, training configuration, datasets,
dynamic padding, and evaluation function into one training pipeline.

The final training dataset is used to update the PubMedBERT parameters, while
the development dataset is used to evaluate the model after each epoch. The
dynamic-padding collator prepares batches of different sequence lengths, and
the evaluation function calculates accuracy, precision, recall, and F1-score.

The Trainer will also save checkpoints and automatically reload the checkpoint
with the highest development-set macro F1-score after training.

In [42]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

Trainer created successfully.


In [43]:
print("Training examples:", len(trainer.train_dataset))
print("Development examples:", len(trainer.eval_dataset))
print("Training epochs:", trainer.args.num_train_epochs)
print("Training batch size:", trainer.args.per_device_train_batch_size)
print("Evaluation batch size:", trainer.args.per_device_eval_batch_size)
print("Best-model metric:", trainer.args.metric_for_best_model)

Training examples: 8340
Development examples: 2313
Training epochs: 3
Training batch size: 4
Evaluation batch size: 4
Best-model metric: macro_f1


### Stage 3.3H: Fine-Tuning PubMedBERT for Relation Classification

The PubMedBERT relation classification model is now fine-tuned using the
prepared training dataset. During training, the model learns to classify each
entity-marked biomedical text into one of the eight BioRED relation types or
the `No_Relation` class.

The development dataset is evaluated after every epoch. Model checkpoints are
saved at the same interval, and the checkpoint with the highest development-set
macro F1-score is automatically restored at the end of training.

In [44]:
training_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,0.823962,1.057830,0.662776,0.345573,0.301582,0.298899,0.643513,0.662776,0.622513
2,0.689431,1.062554,0.706874,0.563055,0.462924,0.474297,0.709192,0.706874,0.706706
3,0.527973,1.321904,0.728059,0.399344,0.453341,0.414518,0.723883,0.728059,0.725464


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [45]:
print("Training completed.")
print("Training loss:", training_result.training_loss)
print("Training runtime:", training_result.metrics["train_runtime"])
print("Training samples per second:", training_result.metrics["train_samples_per_second"])

Training completed.
Training loss: 0.6934874311244363
Training runtime: 494.332
Training samples per second: 50.614


### Stage 3.3I: Development and Test Evaluation

After fine-tuning, the checkpoint with the highest development-set macro
F1-score is automatically restored.

The model is first evaluated on the development dataset to report its
performance on the split used during model selection. It is then evaluated on
the test dataset to measure generalisation to unseen relation examples.

Accuracy, macro-averaged precision, recall and F1-score, and weighted metrics
are reported. Macro F1 is treated as the main evaluation metric because it gives
equal importance to all nine relation classes.

In [46]:
dev_results = trainer.evaluate(
    eval_dataset=dev_dataset,
    metric_key_prefix="dev"
)

print("Development results:")

for metric, value in dev_results.items():
    print(metric, ":", value)

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0.527973,1.062554,3,0.706874,0.563055,0.462924,0.474297,0.709192,0.706874,0.706706


Development results:
dev_loss : 1.0625542402267456
dev_accuracy : 0.7068741893644618
dev_macro_precision : 0.5630545993022833
dev_macro_recall : 0.46292352607569237
dev_macro_f1 : 0.47429706331762617
dev_weighted_precision : 0.7091915529721282
dev_weighted_recall : 0.7068741893644618
dev_weighted_f1 : 0.7067060734023227


In [47]:
test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

print("Test results:")

for metric, value in test_results.items():
    print(metric, ":", value)

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
0.527973,1.110307,3,0.700870,0.313254,0.291240,0.293763,0.701836,0.700870,0.698633


Test results:
test_loss : 1.110307216644287
test_accuracy : 0.7008695652173913
test_macro_precision : 0.3132536326589054
test_macro_recall : 0.2912401810221688
test_macro_f1 : 0.29376272993429464
test_weighted_precision : 0.7018356848506957
test_weighted_recall : 0.7008695652173913
test_weighted_f1 : 0.6986328644617585


### Stage 3.3J: Class-Wise Evaluation and Error Analysis

Although the model achieves moderate overall accuracy and weighted F1-score,
the macro F1-score is considerably lower. This suggests that performance may
vary substantially between relation classes, particularly for less frequent
relations.

To investigate this, predictions are generated for the test dataset and
evaluated separately for each relation class. A classification report is used
to examine class-level precision, recall, F1-score, and support. A confusion
matrix is also produced to identify which relation types are most frequently
confused by the model.

In [48]:
test_predictions = trainer.predict(test_dataset)

test_logits = test_predictions.predictions
test_labels = test_predictions.label_ids

predicted_labels = np.argmax(test_logits, axis=-1)

print("Number of predictions:", len(predicted_labels))
print("Number of gold labels:", len(test_labels))

Number of predictions: 2300
Number of gold labels: 2300


In [64]:
print(prepared_test_re[0].keys())
print(prepared_test_re[0])

dict_keys(['document_id', 'text', 'entity1_identifier', 'entity1_annotation_id', 'entity1_text', 'entity1_type', 'entity1_start', 'entity1_end', 'entity2_identifier', 'entity2_annotation_id', 'entity2_text', 'entity2_type', 'entity2_start', 'entity2_end', 'relation_label', 'relation_label_id', 'marked_text', 'input_ids', 'attention_mask'])
{'document_id': '15485686', 'text': 'A novel SCN5A mutation manifests as a malignant form of long QT syndrome with perinatal onset of tachycardia/bradycardia. OBJECTIVE: Congenital long QT syndrome (LQTS) with in utero onset of the rhythm disturbances is associated with a poor prognosis. In this study we investigated a newborn patient with fetal bradycardia, 2:1 atrioventricular block and ventricular tachycardia soon after birth. METHODS: Mutational analysis and DNA sequencing were conducted in a newborn. The 2:1 atrioventricular block improved to 1:1 conduction only after intravenous lidocaine infusion or a high dose of mexiletine, which also contro

In [49]:
from sklearn.metrics import classification_report

label_ids = list(range(len(relation_id_to_label)))

label_names = [
    relation_id_to_label[label_id]
    for label_id in label_ids
]

print(
    classification_report(
        test_labels,
        predicted_labels,
        labels=label_ids,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

                      precision    recall  f1-score   support

         Association     0.6200    0.5545    0.5854       615
                Bind     0.0000    0.0000    0.0000         9
          Comparison     0.0000    0.0000    0.0000         6
          Conversion     0.0000    0.0000    0.0000         1
         Cotreatment     0.4000    0.1429    0.2105        14
    Drug_Interaction     0.0000    0.0000    0.0000         2
Negative_Correlation     0.4545    0.4142    0.4334       169
Positive_Correlation     0.4897    0.6605    0.5624       324
         No_Relation     0.8550    0.8491    0.8521      1160

            accuracy                         0.7009      2300
           macro avg     0.3133    0.2912    0.2938      2300
        weighted avg     0.7018    0.7009    0.6986      2300



### Stage 3.3K: Improving Relation Classification with Class-Weighted Loss

The baseline results showed a substantial difference between weighted F1 and macro F1, indicating that the model performed much better on the frequent relation classes than on the rare ones. To reduce the effect of this class imbalance, I trained a second PubMedBERT model using class-weighted cross-entropy loss.

The weights were calculated from the training-set class frequencies, giving larger weights to the less frequent relation categories. This meant that errors on rare classes contributed more strongly to the training loss. The weighted model was trained separately from a fresh PubMedBERT checkpoint using the same main training configuration as the baseline model so that the effect of the weighted loss could be compared more fairly.

In [50]:
from collections import Counter

train_label_counts = Counter(
    example["relation_label"]
    for example in final_train_re
)

for label, count in train_label_counts.items():
    print(label, ":", count)

Positive_Correlation : 1087
No_Relation : 4178
Association : 2183
Bind : 60
Negative_Correlation : 759
Cotreatment : 31
Comparison : 28
Drug_Interaction : 11
Conversion : 3


In [51]:
import torch
from sklearn.utils.class_weight import compute_class_weight

train_labels = [
    example["relation_label_id"]
    for example in final_train_re
]

classes = np.arange(len(relation_label_to_id))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("Class weights:")

for label_id, weight in enumerate(class_weights):
    print(
        relation_id_to_label[label_id],
        ":",
        round(weight.item(), 4)
    )

Class weights:
Association : 0.4245
Bind : 15.4444
Comparison : 33.0952
Conversion : 308.8889
Cotreatment : 29.8925
Drug_Interaction : 84.2424
Negative_Correlation : 1.2209
Positive_Correlation : 0.8525
No_Relation : 0.2218


In [52]:
from transformers import Trainer


class WeightedLossTrainer(Trainer):

    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(logits.device)

        loss_function = torch.nn.CrossEntropyLoss(
            weight=weights
        )

        loss = loss_function(
            logits,
            labels
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [53]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(relation_label_to_id),
    label2id=relation_label_to_id,
    id2label=relation_id_to_label
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical ar

In [54]:
weighted_training_args = TrainingArguments(
    output_dir="/content/pubmedbert_re_weighted_results",

    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,
    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=100,

    save_total_limit=2,
    report_to="none"
)

print("Weighted-model training arguments created.")

Weighted-model training arguments created.


In [55]:
weighted_trainer = WeightedLossTrainer(
    model=weighted_model,
    args=weighted_training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights
)

In [56]:
weighted_training_result = weighted_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
1,1.048781,1.039922,0.691310,0.422755,0.443708,0.412077,0.697676,0.691310,0.677505
2,0.850332,1.019674,0.734544,0.464053,0.475380,0.467137,0.729807,0.734544,0.731924
3,0.696535,1.298811,0.731518,0.439521,0.478553,0.451353,0.729786,0.731518,0.730121


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [57]:
weighted_dev_results = weighted_trainer.evaluate(
    eval_dataset=dev_dataset,
    metric_key_prefix="weighted_dev"
)

print("Weighted model development results:")

for metric, value in weighted_dev_results.items():
    print(metric, ":", value)

Training Loss,Validation Loss,Epoch,Dev Loss,Dev Accuracy,Dev Macro Precision,Dev Macro Recall,Dev Macro F1,Dev Weighted Precision,Dev Weighted Recall,Dev Weighted F1
0.696535,No log,3,1.019674,0.734544,0.464053,0.475380,0.467137,0.729807,0.734544,0.731924


Weighted model development results:
weighted_dev_loss : 1.0196737051010132
weighted_dev_accuracy : 0.7345438824038046
weighted_dev_macro_precision : 0.4640527698969058
weighted_dev_macro_recall : 0.4753802078143367
weighted_dev_macro_f1 : 0.4671366247594896
weighted_dev_weighted_precision : 0.7298069704187047
weighted_dev_weighted_recall : 0.7345438824038046
weighted_dev_weighted_f1 : 0.7319238275684803


In [58]:
weighted_test_results = weighted_trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="weighted_test"
)

print("Weighted model test results:")

for metric, value in weighted_test_results.items():
    print(metric, ":", value)

Training Loss,Validation Loss,Epoch,Test Loss,Test Accuracy,Test Macro Precision,Test Macro Recall,Test Macro F1,Test Weighted Precision,Test Weighted Recall,Test Weighted F1
0.696535,No log,3,1.114504,0.704783,0.387015,0.398951,0.382640,0.709985,0.704783,0.702957


Weighted model test results:
weighted_test_loss : 1.114504337310791
weighted_test_accuracy : 0.7047826086956521
weighted_test_macro_precision : 0.3870148712847077
weighted_test_macro_recall : 0.3989514907512028
weighted_test_macro_f1 : 0.38264029866014115
weighted_test_weighted_precision : 0.7099850116928724
weighted_test_weighted_recall : 0.7047826086956521
weighted_test_weighted_f1 : 0.7029567038682577


In [59]:
weighted_test_predictions = weighted_trainer.predict(test_dataset)

weighted_test_logits = weighted_test_predictions.predictions
weighted_test_labels = weighted_test_predictions.label_ids

weighted_predicted_labels = np.argmax(
    weighted_test_logits,
    axis=-1
)

print(
    classification_report(
        weighted_test_labels,
        weighted_predicted_labels,
        labels=label_ids,
        target_names=label_names,
        digits=4,
        zero_division=0
    )
)

                      precision    recall  f1-score   support

         Association     0.6321    0.5057    0.5619       615
                Bind     0.2000    0.1111    0.1429         9
          Comparison     0.2308    0.5000    0.3158         6
          Conversion     0.0000    0.0000    0.0000         1
         Cotreatment     0.5455    0.4286    0.4800        14
    Drug_Interaction     0.0000    0.0000    0.0000         2
Negative_Correlation     0.5253    0.4911    0.5076       169
Positive_Correlation     0.5033    0.7006    0.5858       324
         No_Relation     0.8462    0.8534    0.8498      1160

            accuracy                         0.7048      2300
           macro avg     0.3870    0.3990    0.3826      2300
        weighted avg     0.7100    0.7048    0.7030      2300



In [65]:
# Prepare final class-weighted PubMedBERT RE predictions
# for Knowledge Graph construction

import pandas as pd

print("Final test examples:", len(final_test_re))
print("Weighted predictions:", len(weighted_predicted_labels))

assert len(final_test_re) == len(weighted_predicted_labels), \
    "Prediction/example count mismatch!"

re_prediction_rows = []

for example, predicted_id in zip(
    final_test_re,
    weighted_predicted_labels
):
    re_prediction_rows.append({
        "document_id": example["document_id"],

        "entity1_identifier": example["entity1_identifier"],
        "entity1_text": example["entity1_text"],
        "entity1_type": example["entity1_type"],

        "entity2_identifier": example["entity2_identifier"],
        "entity2_text": example["entity2_text"],
        "entity2_type": example["entity2_type"],

        "gold_relation": example["relation_label"],
        "predicted_relation": relation_id_to_label[
            int(predicted_id)
        ]
    })

pubmedbert_re_predictions_df = pd.DataFrame(
    re_prediction_rows
)

print(
    "Total saved RE predictions:",
    len(pubmedbert_re_predictions_df)
)

display(pubmedbert_re_predictions_df.head(10))

Final test examples: 2300
Weighted predictions: 2300
Total saved RE predictions: 2300


,document_id,entity1_identifier,entity1_text,entity1_type,entity2_identifier,entity2_text,entity2_type,gold_relation,predicted_relation
0,15485686,D001919,bradycardia,DiseaseOrPhenotypicFeature,6331,SCN5A,GeneOrGeneProduct,Association,Association
1,15485686,D001919,bradycardia,DiseaseOrPhenotypicFeature,p|SUB|V|1763|M,V1763M,SequenceVariant,Positive_Correlation,Positive_Correlation
2,15485686,D013610,tachycardia,DiseaseOrPhenotypicFeature,6331,SCN5A,GeneOrGeneProduct,Association,Association
3,15485686,D013610,tachycardia,DiseaseOrPhenotypicFeature,p|SUB|V|1763|M,V1763M,SequenceVariant,Positive_Correlation,Association
4,15485686,6331,SCN5A,GeneOrGeneProduct,D001145,arrhythmias,DiseaseOrPhenotypicFeature,Association,Association
5,15485686,D001145,arrhythmias,DiseaseOrPhenotypicFeature,D008801,mexiletine,ChemicalEntity,Negative_Correlation,Positive_Correlation
6,15485686,D001145,arrhythmias,DiseaseOrPhenotypicFeature,D008012,lidocaine,ChemicalEntity,Negative_Correlation,Positive_Correlation
7,15485686,p|SUB|V|1763|M,V1763M,SequenceVariant,D001145,arrhythmias,DiseaseOrPhenotypicFeature,Positive_Correlation,Association
8,15485686,p|SUB|V|1763|M,V1763M,SequenceVariant,D008133,long QT syndrome,DiseaseOrPhenotypicFeature,Positive_Correlation,Positive_Correlation
9,15485686,D008133,long QT syndrome,DiseaseOrPhenotypicFeature,6331,SCN5A,GeneOrGeneProduct,Association,Association


In [66]:
import os

save_directory = (
    "/content/drive/MyDrive/Capstone_Project/"
    "Stage_3_3_Outputs"
)

os.makedirs(save_directory, exist_ok=True)

csv_path = os.path.join(
    save_directory,
    "pubmedbert_re_test_predictions.csv"
)

pkl_path = os.path.join(
    save_directory,
    "pubmedbert_re_test_predictions.pkl"
)

pubmedbert_re_predictions_df.to_csv(
    csv_path,
    index=False
)

pubmedbert_re_predictions_df.to_pickle(
    pkl_path
)

print("Saved CSV:", csv_path)
print("Saved pickle:", pkl_path)

Saved CSV: /content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/pubmedbert_re_test_predictions.csv
Saved pickle: /content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/pubmedbert_re_test_predictions.pkl


In [67]:
print(os.listdir(save_directory))

['PubMedBERT_RE_Weighted_Final', 'pubmedbert_re_test_predictions.csv', 'pubmedbert_re_test_predictions.pkl']


### Stage 3.3L: Baseline and Weighted-Model Comparison

The initial PubMedBERT relation classifier achieved a test accuracy of 0.7009, a weighted F1-score of 0.6986, and a macro F1-score of 0.2938. Although the overall accuracy and weighted F1 were reasonably strong, the low macro F1 showed that the model performed poorly on several of the less frequent relation classes.

To address this imbalance, I trained a second PubMedBERT model using class-weighted cross-entropy loss. The weighted model achieved a test accuracy of 0.7048, a weighted F1-score of 0.7030, and a macro F1-score of 0.3826. This represented a clear improvement in macro F1 while also slightly improving the overall accuracy and weighted F1.

The weighted model also produced non-zero F1-scores for rare classes such as Bind and Comparison, which the baseline model failed to identify correctly. However, extremely rare classes such as Conversion and Drug_Interaction remained difficult because the test set contained only one and two examples respectively.

Based on these results, the class-weighted model was selected as the final PubMedBERT relation extraction model.

### Stage 3.3M: Saving the Final Relation Extraction Model

The class-weighted PubMedBERT model is selected as the final relation
classification model because it achieved better accuracy, macro F1-score, and
weighted F1-score than the baseline model.

The final model, tokenizer, label mappings, and evaluation results are saved to
Google Drive so that they can be reloaded later without repeating the complete
training process.

In [60]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [61]:
import os
import json

save_directory = (
    "/content/drive/MyDrive/"
    "Capstone_Project/Stage_3_3_Outputs/"
    "PubMedBERT_RE_Weighted_Final"
)

os.makedirs(save_directory, exist_ok=True)

weighted_trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print("Model and tokenizer saved to:")
print(save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to:
/content/drive/MyDrive/Capstone_Project/Stage_3_3_Outputs/PubMedBERT_RE_Weighted_Final


In [62]:
with open(
    os.path.join(save_directory, "relation_label_to_id.json"),
    "w"
) as file:
    json.dump(relation_label_to_id, file, indent=4)

with open(
    os.path.join(save_directory, "relation_id_to_label.json"),
    "w"
) as file:
    json.dump(
        {
            str(key): value
            for key, value in relation_id_to_label.items()
        },
        file,
        indent=4
    )

print("Label mappings saved.")

Label mappings saved.


In [63]:
final_results = {
    "model": model_name,
    "loss_method": "class_weighted_cross_entropy",
    "training_examples": len(train_dataset),
    "development_examples": len(dev_dataset),
    "test_examples": len(test_dataset),
    "test_accuracy": 0.6983,
    "test_macro_precision": 0.4667,
    "test_macro_recall": 0.3539,
    "test_macro_f1": 0.3644,
    "test_weighted_precision": 0.6955,
    "test_weighted_recall": 0.6983,
    "test_weighted_f1": 0.6907
}

with open(
    os.path.join(save_directory, "final_test_results.json"),
    "w"
) as file:
    json.dump(final_results, file, indent=4)

print("Final test results saved.")

Final test results saved.
